In [1]:
import hashlib
import tempfile
from datetime import datetime
from pathlib import Path

import polars as pl

from membership_pipeline import config, ingest, transform, validate, output

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(38)

DEMO_BATCH = """name,email,date_of_birth,mobile_no
Mr. William Dixon,William_Dixon@provider.com,1986/01/10,4060 1711
Sherry Gonzalez DDS,Sherry_Gonzalez@caldwell.net,14-03-1973,66744895
Kristen Horn,Kristen_Horn@lin.com,1974-09-10,737931
Cathy Werner,Cathy_Werner@martinez.net,2018-09-25,71380411
Mary Ball,Mary_Ball@stevens.biz,02/05/1968,88635912
Lori Torres,Lori_Torres@hill.net,1996/02/31,57951752
,Nobody_Here@jackson.com,1990-01-01,12345678
Kimberly Chang,Kimberly_Chang@johnson-lopez.biz,02/27/1974,2692047
"""

DEMO_HOME = Path(tempfile.mkdtemp())
DEMO_INPUT = DEMO_HOME / "input_batches"
DEMO_INPUT.mkdir()
(DEMO_INPUT / "demo_batch.csv").write_text(DEMO_BATCH, encoding="utf-8")

files = ingest.discover_batches(DEMO_INPUT)
raw = ingest.scan_batches(files)

print(type(raw))
print(raw.collect_schema())
print(raw.drop("source_file").collect())

<class 'polars.lazyframe.frame.LazyFrame'>
Schema({'name': String, 'email': String, 'birthday': String, 'mobile': String, 'source_file': String})
shape: (8, 4)
┌─────────────────────┬──────────────────────────────────┬────────────┬───────────┐
│ name                ┆ email                            ┆ birthday   ┆ mobile    │
│ ---                 ┆ ---                              ┆ ---        ┆ ---       │
│ str                 ┆ str                              ┆ str        ┆ str       │
╞═════════════════════╪══════════════════════════════════╪════════════╪═══════════╡
│ Mr. William Dixon   ┆ William_Dixon@provider.com       ┆ 1986/01/10 ┆ 4060 1711 │
│ Sherry Gonzalez DDS ┆ Sherry_Gonzalez@caldwell.net     ┆ 14-03-1973 ┆ 66744895  │
│ Kristen Horn        ┆ Kristen_Horn@lin.com             ┆ 1974-09-10 ┆ 737931    │
│ Cathy Werner        ┆ Cathy_Werner@martinez.net        ┆ 2018-09-25 ┆ 71380411  │
│ Mary Ball           ┆ Mary_Ball@stevens.biz            ┆ 02/05/1968 ┆ 88635912  │


In [2]:
cleaned = transform.clean(raw)
cleaned_df = cleaned.collect()

print("NAME ->")
print(cleaned_df.select("name", "first_name", "last_name"))

print("\nBIRTHDAY ->")
print(cleaned_df.select("birthday", "birthday_date", "birthday_ymd", "age", "above_18"))

print("\nMOBILE ->")
print(cleaned_df.select("mobile", "mobile_normalised"))

NAME ->
shape: (8, 3)
┌─────────────────────┬────────────┬───────────┐
│ name                ┆ first_name ┆ last_name │
│ ---                 ┆ ---        ┆ ---       │
│ str                 ┆ str        ┆ str       │
╞═════════════════════╪════════════╪═══════════╡
│ Mr. William Dixon   ┆ William    ┆ Dixon     │
│ Sherry Gonzalez DDS ┆ Sherry     ┆ Gonzalez  │
│ Kristen Horn        ┆ Kristen    ┆ Horn      │
│ Cathy Werner        ┆ Cathy      ┆ Werner    │
│ Mary Ball           ┆ Mary       ┆ Ball      │
│ Lori Torres         ┆ Lori       ┆ Torres    │
│ null                ┆ null       ┆ null      │
│ Kimberly Chang      ┆ Kimberly   ┆ Chang     │
└─────────────────────┴────────────┴───────────┘

BIRTHDAY ->
shape: (8, 5)
┌────────────┬───────────────┬──────────────┬──────┬──────────┐
│ birthday   ┆ birthday_date ┆ birthday_ymd ┆ age  ┆ above_18 │
│ ---        ┆ ---           ┆ ---          ┆ ---  ┆ ---      │
│ str        ┆ date          ┆ str          ┆ i32  ┆ bool     │
╞════════

In [3]:
print("FAILED RULES (True = this rule rejected the row)")
print(cleaned_df.select(
    pl.col("name"),
    **{reason: ~passes for reason, passes in validate.predicates()},
))

validated = validate.validate(cleaned)
validated_df = validated.collect()

print("\nVERDICT")
print(validated_df.select("name", "is_successful", "rejection_reason", "membership_id"))

FAILED RULES (True = this rule rejected the row)
shape: (8, 6)
┌─────────────────────┬──────────────┬──────────────────────┬──────────┬────────────────┬───────────────┐
│ name                ┆ missing_name ┆ unparseable_birthday ┆ under_18 ┆ invalid_mobile ┆ invalid_email │
│ ---                 ┆ ---          ┆ ---                  ┆ ---      ┆ ---            ┆ ---           │
│ str                 ┆ bool         ┆ bool                 ┆ bool     ┆ bool           ┆ bool          │
╞═════════════════════╪══════════════╪══════════════════════╪══════════╪════════════════╪═══════════════╡
│ Mr. William Dixon   ┆ false        ┆ false                ┆ false    ┆ false          ┆ false         │
│ Sherry Gonzalez DDS ┆ false        ┆ false                ┆ false    ┆ false          ┆ false         │
│ Kristen Horn        ┆ false        ┆ false                ┆ false    ┆ true           ┆ false         │
│ Cathy Werner        ┆ false        ┆ false                ┆ true     ┆ false          ┆

In [4]:
for name, last, ymd, mid in validated_df.filter(pl.col("is_successful")).select(
    "name", "last_name", "birthday_ymd", "membership_id"
).iter_rows():
    digest = hashlib.sha256(ymd.encode()).hexdigest()
    print(f"{name:<22} {last} + sha256({ymd})[:5] = {digest[:5]}  ->  {mid}")
    print(f"{'':<22} full digest: {digest}")

Mr. William Dixon      Dixon + sha256(19860110)[:5] = 3864b  ->  Dixon_3864b
                       full digest: 3864b579395a2aa05673fc20b529368e8d96012c1935514bc24d6f60e9aa068b
Sherry Gonzalez DDS    Gonzalez + sha256(19730314)[:5] = 345f8  ->  Gonzalez_345f8
                       full digest: 345f83d1193dca8d987de5cd515eef1f1758d422407d5ead1fcd2503987e0239


In [5]:
config.configure_logging("INFO")

RUN_TS = datetime(2022, 3, 1, 9, 0, 0)
summary = output.write_outputs(validated, DEMO_HOME / "output", run_ts=RUN_TS)
archived = output.archive_batches(files, DEMO_HOME / "archive", RUN_TS)

print(f"\nrows_in={summary.rows_in} "
      f"successful={summary.successful} unsuccessful={summary.unsuccessful}")

for kind, path in summary.output_paths.items():
    print(f"\n{kind.upper()}  ->  {path.name}")
    print(pl.read_csv(path, infer_schema=False).drop("source_file", strict=False))

print("\narchived:", [p.name for p in archived])
print("input_batches now:", [p.name for p in DEMO_INPUT.glob("*.csv")])

2026-09-19 00:51:43,985 | INFO     | membership_pipeline.output | Wrote 2 row(s) to C:\Users\rahul\AppData\Local\Temp\tmpz3q6lj1y\output\successful\successful_applications_20220301_090000.csv


2026-09-19 00:51:43,987 | INFO     | membership_pipeline.output | Wrote 6 row(s) to C:\Users\rahul\AppData\Local\Temp\tmpz3q6lj1y\output\unsuccessful\unsuccessful_applications_20220301_090000.csv


2026-09-19 00:51:43,989 | INFO     | membership_pipeline.output | Archived 1 batch file(s) to C:\Users\rahul\AppData\Local\Temp\tmpz3q6lj1y\archive\20220301_090000



rows_in=8 successful=2 unsuccessful=6

SUCCESSFUL  ->  successful_applications_20220301_090000.csv
shape: (2, 7)
┌────────────────┬────────────┬───────────┬──────────────────────────────┬──────────┬──────────┬──────────┐
│ membership_id  ┆ first_name ┆ last_name ┆ email                        ┆ birthday ┆ above_18 ┆ mobile   │
│ ---            ┆ ---        ┆ ---       ┆ ---                          ┆ ---      ┆ ---      ┆ ---      │
│ str            ┆ str        ┆ str       ┆ str                          ┆ str      ┆ str      ┆ str      │
╞════════════════╪════════════╪═══════════╪══════════════════════════════╪══════════╪══════════╪══════════╡
│ Dixon_3864b    ┆ William    ┆ Dixon     ┆ William_Dixon@provider.com   ┆ 19860110 ┆ true     ┆ 40601711 │
│ Gonzalez_345f8 ┆ Sherry     ┆ Gonzalez  ┆ Sherry_Gonzalez@caldwell.net ┆ 19730314 ┆ true     ┆ 66744895 │
└────────────────┴────────────┴───────────┴──────────────────────────────┴──────────┴──────────┴──────────┘

UNSUCCESSFUL  ->  uns